# 05 — Produção por nome de autor
Acompanho a transformação dos artigos em autorias, as contagens, o ranking, o recorte dos 15%
e o sorteio de 50 mil nomes. Os dois HTMLs usam as mesmas contagens completas.
O universo é o conjunto atual de textos utilizáveis criado no notebook 04.
Grupos e instituições permanecem; nomes não são desambiguados em pessoas.

In [1]:
from pathlib import Path
import sys
import os
import json
import math
import hashlib
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import FileLink, display

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root))
from src import config as cfg
from src.autores import split_authors
from src.io_utils import atomic_json, atomic_parquet
from src.viz import export_author_streamgraph

REPROCESSAR = False
cfg.ensure_directories()
pasta = cfg.CURATED / "autores"
reconstruir = REPROCESSAR or not (pasta / "autores.parquet").is_file()

## 1. Ligar os textos atuais aos metadados de publicação
A junção por ID garante que cada nome e data pertencem ao artigo correto.
Exibo até cinco artigos para acompanhar a transformação.

In [2]:
ids = pd.read_parquet(cfg.STAGED / "06_ordem_documentos.parquet", columns=["cord_uid"])
metadados = pd.read_parquet(cfg.STAGED / "04_corpus_periodo.parquet",
    columns=["cord_uid", "authors", "ano_publicacao", "data_publicacao", "precisao_data"])
base = ids.merge(metadados, on="cord_uid", how="left", validate="one_to_one", indicator=True)
assert base["_merge"].eq("both").all(), "Um ID não tem metadados de publicação."
exemplo = base.head(5)
display(exemplo[["cord_uid", "authors", "ano_publicacao"]])
display(split_authors(exemplo).head(25))

,cord_uid,authors,ano_publicacao
0,f9tg6xsg,"Nakashima, Tsuyoshi; Miyamoto, Kyohei; Shima, ...",2020
1,f73c639r,"Akohov, Aleksej; Barner, Christoph; Grimmer, S...",2020
2,1qgpa45q,"Kamau, Everlyn; Oketch, John W.; de Laurent, Z...",2020
3,g34f5w6b,"Lepape, Alain; Jean, Astrid; De Waele, Jan; Fr...",2020
4,d1pd09zj,"Mettu, Ravinder; Chen, Chiang-Yun; Wu, Chung-Yi",2020


,cord_uid,autor_chave,nome_original,ano_publicacao,periodo_mensal
0,f9tg6xsg,"nakashima, tsuyoshi","Nakashima, Tsuyoshi",2020,2020-01
1,f9tg6xsg,"miyamoto, kyohei","Miyamoto, Kyohei",2020,2020-01
2,f9tg6xsg,"shima, nozomu","Shima, Nozomu",2020,2020-01
3,f9tg6xsg,"kato, seiya","Kato, Seiya",2020,2020-01
4,f9tg6xsg,"kawazoe, yu","Kawazoe, Yu",2020,2020-01
5,f9tg6xsg,"ohta, yoshinori","Ohta, Yoshinori",2020,2020-01
6,f9tg6xsg,"morimoto, takeshi","Morimoto, Takeshi",2020,2020-01
7,f9tg6xsg,"yamamura, hitoshi","Yamamura, Hitoshi",2020,2020-01
8,f73c639r,"akohov, aleksej","Akohov, Aleksej",2020,2020-01
9,f73c639r,"barner, christoph","Barner, Christoph",2020,2020-01


## 2. Separar as autorias em blocos
Cada nome recebe uma publicação inteira por artigo. Divido somente por ponto e vírgula,
normalizo Unicode/espaços/maiúsculas e retiro repetições do mesmo nome no mesmo artigo.
Acumulo contagens anuais e variantes de escrita; não crio uma visualização mensal.

In [3]:
if reconstruir:
    anuais_partes, variantes_partes = [], []
    escritor = None
    try:
        for inicio in range(0, len(base), 5000):
            pares = split_authors(base.iloc[inicio:inicio + 5000])
            tabela = pa.Table.from_pandas(pares, preserve_index=False)
            if escritor is None:
                escritor = pq.ParquetWriter(cfg.STAGED / "12_autorias.parquet", tabela.schema, compression="zstd")
            escritor.write_table(tabela)
            anuais_partes.append(pares.groupby(["autor_chave", "ano_publicacao"], as_index=False).agg(
                publicacoes=("cord_uid", "size"), publicacoes_com_mes=("periodo_mensal", "count")))
            variantes_partes.append(pares.groupby(["autor_chave", "nome_original"], as_index=False).size())
    finally:
        if escritor is not None:
            escritor.close()

## 3. Somar as contagens e escolher o nome de exibição
Consolido os blocos. O nome exibido é a variante mais frequente, com desempate lexical.
As datas somente anuais entram no total e ficam identificadas como sem mês conhecido.

In [4]:
if reconstruir:
    anual = pd.concat(anuais_partes, ignore_index=True).groupby(
        ["autor_chave", "ano_publicacao"], as_index=False)[["publicacoes", "publicacoes_com_mes"]].sum()
    variantes = pd.concat(variantes_partes, ignore_index=True).groupby(
        ["autor_chave", "nome_original"], as_index=False)["size"].sum()
    nomes = variantes.sort_values(["size", "nome_original"], ascending=[False, True]).drop_duplicates("autor_chave")
    autores = anual.groupby("autor_chave", as_index=False)[["publicacoes", "publicacoes_com_mes"]].sum()
    autores = autores.rename(columns={"publicacoes": "publicacoes_total"})
    autores["publicacoes_sem_mes"] = autores["publicacoes_total"] - autores["publicacoes_com_mes"]
    autores = autores.merge(nomes[["autor_chave", "nome_original"]], on="autor_chave", validate="one_to_one")
    autores = autores.rename(columns={"nome_original": "autor_nome"})
    autores["autor_id"] = autores["autor_chave"].map(lambda nome: hashlib.sha256(nome.encode()).hexdigest()[:20])
    anual = anual.rename(columns={"ano_publicacao": "periodo"})
    anual["periodo"] = anual["periodo"].astype("string")
    atomic_parquet(pasta / "publicacoes_anuais.parquet", anual[["autor_chave", "periodo", "publicacoes"]])
    del anuais_partes, variantes_partes, variantes
else:
    autores = pd.read_parquet(pasta / "autores.parquet")
    anual = pd.read_parquet(pasta / "publicacoes_anuais.parquet")
display(autores.head())

,autor_chave,publicacoes_total,publicacoes_com_mes,publicacoes_sem_mes,autor_nome,autor_id,posicao,selecionado
0,"ieee,",508,0,508,"Ieee,",185f16c3f658dd2e6eff,1,True
1,"wang, wei",442,408,34,"Wang, Wei",0b600593465a10fd84ec,2,True
2,"zhang, wei",374,343,31,"Zhang, Wei",6abcd017a12b0deda068,3,True
3,"wang, jing",362,336,26,"Wang, Jing",a862caeb506b91987e4c,4,True
4,"liu, yang",350,331,19,"Liu, Yang",0f4821989d5edd9c1e4b,5,True


## 4. Ranquear os nomes e selecionar os 15% mais produtivos
Ordeno por publicações decrescentes e chave crescente. O corte usa o teto de 15% do número
de nomes e não é ampliado por empates. Os CSVs permitem conferir todos os nomes e os selecionados.

In [5]:
autores = autores.sort_values(["publicacoes_total", "autor_chave"], ascending=[False, True]).reset_index(drop=True)
autores["posicao"] = range(1, len(autores) + 1)
quantidade = math.ceil(cfg.AUTHOR_TOP_FRACTION * len(autores))
autores["selecionado"] = autores["posicao"].le(quantidade)
selecionados = autores.loc[autores["selecionado"]].copy()
atomic_parquet(pasta / "autores.parquet", autores)
for nome, tabela in [("autores_ranking_completo", autores), ("autores_top15_porcento", selecionados),
                      ("autores_top25", selecionados.head(cfg.AUTHOR_DEFAULT_VISIBLE))]:
    tabela.to_csv(cfg.OUTPUTS / "tables" / f"{nome}.csv", index=False, encoding="utf-8-sig")
metodologia = {"corpus_referencia": len(base), "nomes_identificados": len(autores),
               "nomes_selecionados": len(selecionados), "fracao": cfg.AUTHOR_TOP_FRACTION,
               "autorias_unicas": int(autores["publicacoes_total"].sum()),
               "identificacao": "Nomes normalizados, sem desambiguação; inclui instituições e grupos."}
atomic_json(pasta / "metodologia.json", metodologia)
display(selecionados[["posicao", "autor_nome", "publicacoes_total", "publicacoes_sem_mes"]].head(25))
print(f"Nomes: {len(autores):,}; selecionados: {len(selecionados):,}")

,posicao,autor_nome,publicacoes_total,publicacoes_sem_mes
0,1,"Ieee,",508,508
1,2,"Wang, Wei",442,34
2,3,"Zhang, Wei",374,31
3,4,"Wang, Jing",362,26
4,5,"Liu, Yang",350,19
5,6,"Mahase, Elisabeth",345,68
6,7,"Li, Jing",299,22
7,8,"Li, Yan",298,26
8,9,"Li, Wei",284,27
9,10,"Zhang, Li",279,22


Nomes: 1,784,849; selecionados: 267,728


## 5. Sortear a amostra de cinquenta mil nomes
O sorteio é uniforme dentro dos 15% mais produtivos, sem reposição e com seed 42.
Restauro a posição original e mantenho todas as publicações dos nomes sorteados.
Artigos não são sorteados. O CSV registra exatamente os nomes da amostra.

In [6]:
amostra = selecionados.sample(n=cfg.AUTHOR_SAMPLE_SIZE, replace=False, random_state=cfg.RANDOM_SEED)
amostra = amostra.sort_values("posicao").reset_index(drop=True)
amostra.to_csv(cfg.OUTPUTS / "tables/autores_amostra.csv", index=False, encoding="utf-8-sig")
metodo_amostra = {"metodo": "aleatoria_simples_sem_reposicao", "unidade": "nome_normalizado",
                  "nomes_elegiveis": len(selecionados), "nomes_sorteados": len(amostra), "seed": cfg.RANDOM_SEED,
                  "universo": "15% dos nomes com mais publicações", "contagens": "Publicações completas dos nomes sorteados."}
atomic_json(cfg.OUTPUTS / "tables/autores_amostra_metodologia.json", metodo_amostra)
display(amostra[["posicao", "autor_nome", "publicacoes_total"]].head(10))

,posicao,autor_nome,publicacoes_total
0,7,"Li, Jing",299
1,12,"Liu, Wei",267
2,13,"Iacobucci, Gareth",266
3,32,"Li, Y.",195
4,34,"Wang, Y.",189
5,35,"Lippi, Giuseppe",188
6,42,"Li, Hui",183
7,45,"Liu, Jing",179
8,53,"Rezaei, Nima",166
9,61,"Liu, Li",159


## 6. Gerar os dois HTMLs anuais
O final contém todos os nomes selecionados; a amostra contém os cinquenta mil sorteados.
Cada HTML começa com os 25 primeiros nomes do respectivo conjunto e permite filtrá-los.
D3 e dados são incorporados, permitindo abrir sem internet. Não são produzidos PNGs nem SVGs separados.
Há três observações anuais; a curva entre elas é uma interpolação visual.

In [7]:
for nomes, eh_amostra in [(selecionados, False), (amostra, True)]:
    explicacao = {**metodologia, **({"amostra_nomes": metodo_amostra} if eh_amostra else {})}
    html = export_author_streamgraph(nomes, anual, explicacao, sample=eh_amostra)
    display(FileLink(os.path.relpath(html, Path.cwd())))

C:\UTFPR\TCC\tcc-data-viz\tcc-data-viz-main\outputs\completos\streamgraph_autores.html

C:\UTFPR\TCC\tcc-data-viz\tcc-data-viz-main\outputs\amostras\streamgraph_autores.html